### **NOTE**: This was the initial inception of this project, later I switched to bare Python files for modularity and code reproducibility, one can still go through this notebook (with caution) to grasp where I was trying to pitch this project. - Thank You

In [ ]:
from typing import List
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from scipy.integrate import solve_ivp
import pandas as pd
import pickle as pkl
from solver import RK45

# <div align = "center"> Introduction </div>
#### **NOTE**: A clear, intuitive understanding of the system will be added to aid in understanding the motivations of this simulation
- We aim to model a realistic single DOF oscillator, excited by rotating forces and impacts
- Core system equation is afformentioned
<div align = "center"> $m\ddot{x}+c\dot{x} + kx = F_{imb}(t) + F_{fault}(t) + F_{noise}(t)$ </div><br>

- $F_{imb}(t) = m_er\omega^2\sin(\omega t)$ is a rotor with eccentric mass with $\omega = 2\pi f_{shaft}$
- $F_{fault}(t) = \sum_{n=0}^{N}A\delta (t-nT_f)$ where $\delta$ is the dirac delta function and $T_f = \frac{1}{f_{fault}}$ (we may replace this with even realistic impulse having equation of form $h(t) = Ae^{-\beta t}\sin(\omega_n t)$

### Finalized system governing equations
<div align = "center">$m\ddot{x}+c\dot{x} + kx = F_0\sin (\omega t) + \sum_{n=0}^NAe^{-\beta (t-nT_f)} \sin(\omega_n (t-nT_f)) + \epsilon(t)$</div><br>

<div align = "center">$\ddot{x} + 2 \zeta \omega_n\dot{x} + \omega_n^2x = \frac{F_0}{m}\sin(\omega t) + \frac{1}{m}\sum_{n=0}^{N}Ae^{-\beta (t-nT_f)} \sin(\omega_n (t-nT_f))+ \frac{\epsilon(t)}{m}$</div>


### NOTE
- upon careful consideration of the problem we realized that, adding noise in the forcing makes this problem less deterministic and smooth.
- whereas solvers like RK45 are specifically made for deterministic and smooth equations, hence this conflicts directly with the solver
- so we will add noise in signal collection step and not in forcing

<div align = "center">$\ddot{x} + 2 \zeta \omega_n\dot{x} + \omega_n^2x = \frac{F_0}{m}\sin(\omega t) + \frac{1}{m}\sum_{n=0}^{N}Ae^{-\beta (t-nT_f)} \sin(\omega_n (t-nT_f))$</div> <br>
<div align  = "center"> $y(t) = y_{\text{clean}}(t) + \epsilon$</div><br>
<div align = "center">$\epsilon \sim N(0,1)$</div>

In [6]:
from dataclasses import dataclass,field

@dataclass(frozen = True)
class parameters():
    '''
    This class contains the global parameters to be swept upon,
    we'll use this as a base class to get our once fixed parameter ranges and values, 
    another similar class with scalar values will be used for runtime parameter allocation
    '''
    m:float = 1.0 ##mass of the system
    f_n:float = 300#hertz ##natural frequency of the system
    zeta:float = 0.02 #damping coefficient

    f_shaft:np.ndarray = field(default_factory = lambda: np.array([20,30,40]))
    F0:np.ndarray = field(default_factory = lambda: np.array([5,10,20]))

    f_fault_multipliers:np.ndarray = field(default_factory = lambda: np.array([3,5,7]))
    A_base:np.ndarray = field(default_factory = lambda: np.array([20,40,60]))
    severity:np.ndarray = field(default_factory = lambda: np.array([0.3,0.6,1.0]))
    sigma:np.ndarray = field(default_factory = lambda: np.array([0.2,0.5,1.0]))

    dt: float = 1e-4
    T_total:float = 1.5
    # e:np.ndarray = field(default_factory = lambda: np.random.normal(size=T_total/dt))

    def __post_init__(self):
        '''
        This is a specialized additional constructor method of dataclass
        which runs right after an instance of the parent dataclass is made
        '''
        object.__setattr__(self,"omega_n",2*np.pi*self.f_n)
        object.__setattr__(self,"k",self.m*self.omega_n**2)
        object.__setattr__(self,"c",2*self.zeta*np.sqrt(self.k*self.m))
        object.__setattr__(self,"beta",self.zeta*self.omega_n)
        # object.__setattr__(self,"T_f",1/self.f_shaft)



@dataclass
class SimParams():
    m: float
    omega_n: float
    zeta: float
    beta: float
    f_shaft: float
    omega: float
    F0: float
    f_fault: float
    T_f: float
    A: float


def resolver(base:parameters,f_shaft,F0,fault_multi,A_base,severity,sigma):
    '''
    Inside the simulation sweep loop, this function will be called to return the 
    params object.
    '''
    f_fault = fault_multi * f_shaft

    return SimParams(
        m = base.m,
        omega_n = base.omega_n,
        zeta = base.zeta,
        beta = base.beta,

        f_shaft = f_shaft,
        omega = 2*np.pi*f_shaft,
        F0 = F0,

        f_fault = f_fault,
        T_f = 1 / f_fault,
        A = A_base*severity,
    )

In [7]:

'''
This function right now has multiple issues, but this project will follow a general structure afformentioned,
I'll separate the different components of forcing to make this robust.
'''
def imbalance_force(t,params):
    return (params.F0/params.m) * np.sin(params.omega*t)

def fault_force(t,params):
    '''
    This essentially runs for each RK45 step, which can become computationally very expensive for large time simulations
    We'll cap the N based on the value of t and T_f
    '''
    total:float = 0.0
    N:int = int(t / params.T_f)

    for n in range(max(0,N-5),N+1):
        tau = t - n*params.T_f
        if tau > 0:
            total += params.A * np.exp(-params.beta * tau)*np.sin(params.omega_n*tau)

    return total / params.m

# def noise_force(params):
#     return params.sigma * np.random.randn() / params.m
    

def system(t,state,params:SimParams):
    x,xdot = state

    xddot = (
        -2 * params.zeta * params.omega_n * xdot
        - params.omega_n**2 * x
        + imbalance_force(t,params)
        + fault_force(t,params)
    )
    return np.array([xdot,xddot])